# RAG Pipeline — Step by Step

This notebook breaks down a RAG (Retrieval-Augmented Generation) pipeline into individual steps.
Run each cell in order and read the output before moving to the next one.

**To experiment:** change the values in **Step 0**, then re-run all cells from top to bottom.

## Setup

Create a `.env` file in the `rag-demo/` root folder with:
```
GOOGLE_API_KEY=your_key_here
```

Install dependencies:
```bash
pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-google-genai sentence-transformers chromadb rank-bm25 python-dotenv
```

In [ ]:
import sys                                                                                                                     
print(sys.executable)

In [ ]:
import os
import chromadb
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

load_dotenv(dotenv_path=Path("../.env"))
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Imports done.")

## Step 0: Configuration

All the parameters you can tune are here. Change a value and re-run all cells below to see how it affects the output.

In [ ]:
# --- Document ---
DOCUMENT_PATH = "../test_documents"  # path to a file or folder

# --- Chunking ---
CHUNK_SIZE    = 1000   # max characters per chunk
CHUNK_OVERLAP = 200    # how many characters overlap between chunks
SEPARATORS    = ["\n\n", "\n", ". ", " ", ""]  # split priority: try paragraph first, then sentence, etc.

# --- Retrieval ---
QUERY    = "How does retrieval augmented generation work?"
K_VECTOR = 5   # number of chunks to fetch via vector search
K_BM25   = 5   # number of chunks to fetch via BM25 keyword search

# --- Reranking ---
TOP_K = 3      # number of chunks to keep after reranking (these go to the LLM)

# --- Generation ---
GEMINI_MODEL = "gemini-2.5-flash"
TEMPERATURE  = 0.3    # 0 = more deterministic, 1 = more creative
MAX_TOKENS   = 1024

---
## Step 1: Load Documents

Read raw files from disk. This is the unprocessed text before any splitting or cleaning.

In [ ]:
loader = DirectoryLoader(
    DOCUMENT_PATH,
    glob="**/*",
    loader_cls=UnstructuredFileLoader,
    show_progress=True,
)
documents = loader.load()

print(f"Loaded {len(documents)} documents:")
for doc in documents:
    name = Path(doc.metadata["source"]).name
    print(f"  {name} — {len(doc.page_content)} chars")

In [ ]:
# Preview the raw text of the first document
print(documents[0].page_content[:600])

---
## Step 2: Chunking

Split each document into smaller overlapping pieces called chunks.
The LLM cannot read the entire document at once — chunking makes retrieval possible.

Try changing `CHUNK_SIZE` and `CHUNK_OVERLAP` in Step 0 and re-running these cells.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=SEPARATORS,
)

chunks = splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

In [ ]:
# Print the first 3 chunks so you can see where the splits happened
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i + 1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:400])
    print()

---
## Step 3: Embedding

Convert each chunk into a vector — a list of numbers that captures its meaning.
Chunks with similar meaning will have similar vectors.
We use a free local model so there are no API calls or costs.

In [ ]:
# Downloads ~80 MB on first run, then cached locally
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample = embeddings_model.embed_query("test")
print(f"Embedding model loaded. Each chunk becomes a vector of {len(sample)} numbers.")

In [ ]:
# Show what a vector actually looks like for the first chunk
vec = embeddings_model.embed_query(chunks[0].page_content)
print(f"First 10 values of Chunk 1's vector:")
print([round(v, 4) for v in vec[:10]])

---
## Step 4: Build Vector Store

Embed all chunks and store them in ChromaDB so we can search them later.

In [ ]:
# EphemeralClient = in-memory only, safe to re-run without restarting the kernel
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="rag_demo",
    client=chromadb.EphemeralClient(),
)

print(f"Stored {vectorstore._collection.count()} vectors in ChromaDB.")

---
## Step 5: Retrieval

Given a query, find the most relevant chunks. We run two searches — vector and keyword — then merge them.

### 5.1 Vector Search

Converts the query to a vector and finds the closest chunk vectors by similarity.
Good at finding relevant chunks even if they use different words than the query.

In [ ]:
vector_docs = vectorstore.similarity_search(QUERY, k=K_VECTOR)

print(f"Query: {QUERY}")
print(f"\nTop {K_VECTOR} results from vector search:")
for i, doc in enumerate(vector_docs):
    print(f"\n[{i + 1}] {doc.page_content[:300]}")

### 5.2 BM25 (Keyword Search)

Scores chunks by how often the query's keywords appear in them — no vectors involved.
Useful when the query uses the same specific words as the document.

In [ ]:
# Build a BM25 index over all chunks
corpus = [chunk.page_content.lower().split() for chunk in chunks]
bm25   = BM25Okapi(corpus)

# Search
scores      = bm25.get_scores(QUERY.lower().split())
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:K_BM25]
bm25_docs   = [chunks[i] for i in top_indices]

print(f"Query: {QUERY}")
print(f"\nTop {K_BM25} results from BM25:")
for i, (idx, doc) in enumerate(zip(top_indices, bm25_docs)):
    print(f"\n[{i + 1}] score={scores[idx]:.2f}")
    print(doc.page_content[:300])

### 5.3 Hybrid: Reciprocal Rank Fusion (RRF)

Combine the two result lists into one. A chunk that ranks highly in both vector and BM25 search
rises to the top. This is more robust than either search alone.

In [ ]:
def rrf(list_a, list_b, k=60):
    """Merge two ranked lists. Higher-ranked docs in both lists get the highest combined score."""
    scores = {}
    for rank, doc in enumerate(list_a):
        scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (k + rank + 1)
    for rank, doc in enumerate(list_b):
        scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (k + rank + 1)

    all_docs   = {doc.page_content: doc for doc in list_a + list_b}
    sorted_keys = sorted(scores, key=scores.get, reverse=True)
    return [all_docs[key] for key in sorted_keys]


candidates = rrf(vector_docs, bm25_docs)

print(f"{len(candidates)} unique chunks after merging vector + BM25 results:")
for i, doc in enumerate(candidates):
    print(f"\n[{i + 1}] {doc.page_content[:200]}")

---
## Step 6: Reranking

A cross-encoder reads the query and each candidate chunk together to give a more accurate relevance score.
It is slower than vector search, so we only run it on the small candidate set from Step 5.

In [ ]:
# Downloads ~85 MB on first run, then cached locally
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded.")

In [ ]:
# Score each candidate against the query
pairs  = [[QUERY, doc.page_content] for doc in candidates]
scores = cross_encoder.predict(pairs)

# Sort by score descending, keep top K
ranked     = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
final_docs = [doc for doc, score in ranked[:TOP_K]]

print(f"Reranked {len(candidates)} candidates, kept top {TOP_K}:")
for i, (doc, score) in enumerate(ranked[:TOP_K]):
    print(f"\n[{i + 1}] score={score:.2f}")
    print(doc.page_content[:300])

---
## Step 7: Generation

Pass the query and the top chunks to the LLM. We print the full prompt first so you can see
exactly what the model receives before it generates an answer.

In [ ]:
# Build the prompt from the final chunks
context = "\n\n".join(
    f"[{i + 1}] {doc.page_content}"
    for i, doc in enumerate(final_docs)
)

prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {QUERY}

Answer:"""

print(prompt)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

response = llm.invoke(prompt)
print(response.content)